# GPU Import & Test

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.2.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


# DATASET PATH

In [2]:
def find_dataset_path(start_folder):
    for root, dirs, files in os.walk(start_folder):
        if "train" in dirs and "valid" in dirs:
            return root
    return None

base_dir = find_dataset_path("/kaggle/input")

if base_dir is None:
    base_dir = "../input/new-plant-diseases-dataset/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)"
    print("[WARNING] Using fallback path:", base_dir)
else:
    print("[INFO] Dataset found at:", base_dir)


[INFO] Dataset found at: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)


# TRAIN CONFIGURATION

In [3]:
IMAGE_SIZE = 224
BATCH_SIZE = 128
EPOCHS = 10


# DATA GENERATOR

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    os.path.join(base_dir, "train"),
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    os.path.join(base_dir, "valid"),
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_data.num_classes
print("Number of classes:", NUM_CLASSES)


Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.
Number of classes: 38


# MODEL RESNET50

In [5]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

base_model.trainable = False

inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs, outputs)
model.summary()


94773248/94765736 [==============================] - 0s 0us/step
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 224, 224, 3)]     0         
_________________________________________________________________
resnet50 (Model)             (None, 7, 7, 2048)        23587712  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
dense (Dense)                (None, 38)                77862     
Total params: 23,665,574
Trainable params: 77,862
Non-trainable params: 23,587,712
_________________________________________________________________


# COMPILE

In [6]:
optimizer = SGD(learning_rate=0.01, momentum=0.9)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# CALLBACK + SAVE MODEL

In [7]:
SAVE_DIR = "/kaggle/working/models"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, "resnet50_best.h5")

callbacks = [
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]


# TRAIN

In [8]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    steps_per_epoch=len(train_data),
    validation_steps=len(val_data),
    workers=4,
    use_multiprocessing=True,
    callbacks=callbacks
)


Epoch 1/10
550/550 [==============================] - ETA: 0s - loss: 0.6266 - accuracy: 0.8142
Epoch 00001: val_accuracy improved from -inf to 0.94816, saving model to /kaggle/working/models/resnet50_best.h5
550/550 [==============================] - 609s 1s/step - loss: 0.6266 - accuracy: 0.8142 - val_loss: 0.1792 - val_accuracy: 0.9482 - lr: 0.0100
Epoch 2/10
550/550 [==============================] - ETA: 0s - loss: 0.3043 - accuracy: 0.9029
Epoch 00002: val_accuracy improved from 0.94816 to 0.95168, saving model to /kaggle/working/models/resnet50_best.h5
550/550 [==============================] - 576s 1s/step - loss: 0.3043 - accuracy: 0.9029 - val_loss: 0.1575 - val_accuracy: 0.9517 - lr: 0.0100
Epoch 3/10
550/550 [==============================] - ETA: 0s - loss: 0.2583 - accuracy: 0.9176
Epoch 00003: val_accuracy improved from 0.95168 to 0.95948, saving model to /kaggle/working/models/resnet50_best.h5
550/550 [==============================] - 575s 1s/step - loss: 0.2583 - accu

# EVALUATE + SAVE

In [9]:
model.evaluate(val_data)
model.save("/kaggle/working/leaf_disease_resnet50_final.h5")


138/138 [==============================] - 70s 506ms/step - loss: 0.0904 - accuracy: 0.9710


# ZIP OUTPUT (KAGGLE)

In [10]:
!zip -r output_files.zip /kaggle/working/models /kaggle/working/leaf_disease_resnet50_final.h5


  adding: kaggle/working/models/ (stored 0%)
  adding: kaggle/working/models/resnet50_best.h5 (deflated 8%)
  adding: kaggle/working/leaf_disease_resnet50_final.h5 (deflated 8%)
